# Experiment: TravelPlanner Organization-Philosophy Scientific Comparison on OpenRouter Qwen3.5-9B

This notebook is the **principal publication-oriented benchmark notebook** for the thesis.

## Research question

At backbone-constant conditions and under the official TravelPlanner scorer, does the stigmergic organization outperform reproducible centralized or monolithic organizations, and at what operational cost?

## Compared organization philosophies

- **Direct Solo**
- **CoT Solo**
- **Self-Refine Solo**
- **Central Planner-Executor**
- **Central Graph Supervisor**
- **StigmergiAgentic**

## Controlled dimensions

- provider: **OpenRouter**
- model: **`qwen/qwen3.5-9b`**
- split: **`validation`**
- scorer: **official TravelPlanner scorer** from this repository
- output contract: `query_XXX.json -> runs.json -> official_eval.json`
- execution mode: **Docker-first**

## Scientific protocol

- primary endpoint: **Final Pass Rate**
- secondary endpoints: delivery, commonsense micro/macro, hard-constraint micro/macro, cost, time, coordination overhead, reproducibility
- replications: **3 seeds** (`42`, `43`, `44`)
- gating: **preflight -> pilot -> full**
- invalid runs are classified as `infra_failure`, `framework_failure`, or `partial_success`; they are **not** converted into score `0`


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import shlex
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display


def _find_repo_root() -> Path:
    candidate = Path.cwd().resolve()
    for _ in range(6):
        if (candidate / 'main.py').exists():
            return candidate
        candidate = candidate.parent
    raise RuntimeError(f'Cannot find repository root from {Path.cwd()}')


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)


def run_command(
    cmd: list[str],
    *,
    cwd: Path = REPO_ROOT,
    check: bool = True,
    log_path: Path | None = None,
    env: dict[str, str] | None = None,
    live: bool = False,
) -> subprocess.CompletedProcess[str]:
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items()})
    print('$', shlex.join(cmd))
    if live:
        process = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            env=merged_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        chunks: list[str] = []
        assert process.stdout is not None
        for line in process.stdout:
            chunks.append(line)
            print(line, end='')
        process.wait()
        stdout = ''.join(chunks)
        stderr = ''
        returncode = int(process.returncode or 0)
    else:
        proc = subprocess.run(
            cmd,
            cwd=str(cwd),
            env=merged_env,
            capture_output=True,
            text=True,
            check=False,
        )
        stdout = proc.stdout
        stderr = proc.stderr
        returncode = proc.returncode
    combined = stdout + ('' if not stderr else ('\n' + stderr))
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(combined, encoding='utf-8')
    if (not live) and combined.strip():
        print(combined.strip()[:16000])
    if check and returncode != 0:
        raise RuntimeError(f'Command failed with exit={returncode}: {shlex.join(cmd)}')
    return subprocess.CompletedProcess(cmd, returncode, stdout, stderr)


def run_docker_python(
    python_args: list[str],
    *,
    log_path: Path | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    cmd = ['docker', 'compose', 'run', '--rm', '-e', 'PYTHONUNBUFFERED=1', 'travelplanner-smoke', 'python', *python_args]
    return run_command(cmd, cwd=REPO_ROOT, check=check, log_path=log_path, live=True)


def load_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')


def require_path(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def render_markdown_file(path: Path) -> None:
    require_path(path)
    display(Markdown(path.read_text(encoding='utf-8')))


def git_sha() -> str:
    return run_command(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT).stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    digest.update(path.read_bytes())
    return digest.hexdigest()


def ensure_travelplanner_smoke_image(
    *,
    log_path: Path,
    force_build: bool = False,
    skip_build: bool = False,
) -> dict[str, Any]:
    if shutil.which('docker') is None:
        raise RuntimeError(
            'Docker CLI is not available in the current environment. '
            'Install Docker Desktop or relaunch Jupyter from a shell where `docker` is on PATH.'
        )
    tracked_inputs = [
        REPO_ROOT / 'Dockerfile',
        REPO_ROOT / 'docker-compose.yml',
        REPO_ROOT / 'requirements.txt',
    ]
    signature = {
        path.name: sha256_file(path)
        for path in tracked_inputs
        if path.exists()
    }
    cache_path = REPO_ROOT / 'output' / 'docker_cache' / 'travelplanner_scientific_build_state.json'
    cached = load_json(cache_path, {})

    if skip_build:
        print('Skipping Docker build because TRAVELPLANNER_COMPARE_SKIP_DOCKER_BUILD=1.')
        return {'skipped': True, 'reason': 'env_skip'}
    if (not force_build) and isinstance(cached, dict) and cached.get('signature') == signature:
        print('Skipping Docker build: cached image inputs are unchanged.')
        return {'skipped': True, 'reason': 'cached'}

    run_command(
        ['docker', 'compose', 'build', '--progress=plain', 'travelplanner-smoke'],
        cwd=REPO_ROOT,
        log_path=log_path,
        live=True,
    )
    write_json(
        cache_path,
        {
            'signature': signature,
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        },
    )
    return {'skipped': False}


## Environment and Reproducibility


In [2]:
RUN_TAG = os.environ.get('TRAVELPLANNER_COMPARE_RUN_TAG') or datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
MODEL_NAME = os.environ.get('TRAVELPLANNER_COMPARE_MODEL', 'qwen/qwen3.5-9b')
OPENROUTER_BASE_URL = os.environ.get('TRAVELPLANNER_COMPARE_BASE_URL', 'https://openrouter.ai/api/v1')
SPLIT = os.environ.get('TRAVELPLANNER_COMPARE_SPLIT', 'validation')
SEEDS = os.environ.get('TRAVELPLANNER_COMPARE_SEEDS', '42,43,44')
STUDY_ARMS = os.environ.get(
    'TRAVELPLANNER_COMPARE_ARMS',
    'solo_direct,solo_cot,solo_self_refine,planner_executor,langgraph_supervisor,stigmergiagentic',
)
PREFLIGHT_COUNT = int(os.environ.get('TRAVELPLANNER_COMPARE_PREFLIGHT_COUNT', '3'))
PILOT_COUNT = int(os.environ.get('TRAVELPLANNER_COMPARE_PILOT_COUNT', '20'))
FULL_COUNT = int(os.environ.get('TRAVELPLANNER_COMPARE_FULL_COUNT', '180'))
DOCKER_FORCE_BUILD = os.environ.get('TRAVELPLANNER_COMPARE_FORCE_DOCKER_BUILD', '0') == '1'
DOCKER_SKIP_BUILD = os.environ.get('TRAVELPLANNER_COMPARE_SKIP_DOCKER_BUILD', '0') == '1'
RUN_PREFLIGHT = os.environ.get('TRAVELPLANNER_COMPARE_RUN_PREFLIGHT', '1') == '1'
RUN_PILOT = os.environ.get('TRAVELPLANNER_COMPARE_RUN_PILOT', '1') == '1'
RUN_FULL = os.environ.get('TRAVELPLANNER_COMPARE_RUN_FULL', '1') == '1'
BUILD_PACK = os.environ.get('TRAVELPLANNER_COMPARE_BUILD_PACK', '1') == '1'

SOLO_DIRECT_BUDGET = float(os.environ.get('TRAVELPLANNER_COMPARE_SOLO_DIRECT_BUDGET_USD', '20'))
SOLO_COT_BUDGET = float(os.environ.get('TRAVELPLANNER_COMPARE_SOLO_COT_BUDGET_USD', '20'))
SOLO_SELF_REFINE_BUDGET = float(os.environ.get('TRAVELPLANNER_COMPARE_SOLO_SELF_REFINE_BUDGET_USD', '20'))
PLANNER_EXECUTOR_BUDGET = float(os.environ.get('TRAVELPLANNER_COMPARE_PLANNER_EXECUTOR_BUDGET_USD', '20'))
LANGGRAPH_BUDGET = float(os.environ.get('TRAVELPLANNER_COMPARE_LANGGRAPH_BUDGET_USD', '20'))
OUR_BUDGET = float(os.environ.get('TRAVELPLANNER_COMPARE_OUR_BUDGET_USD', '20'))
LANGGRAPH_MAX_VALIDATION_RETRIES = int(os.environ.get('TRAVELPLANNER_COMPARE_LANGGRAPH_MAX_VALIDATION_RETRIES', '2'))
OUR_MAX_TICKS = int(os.environ.get('TRAVELPLANNER_COMPARE_OUR_MAX_TICKS', '30'))
OUR_AGENTS = int(os.environ.get('TRAVELPLANNER_COMPARE_OUR_AGENTS', '3'))

COMPARE_ROOT = REPO_ROOT / 'output' / 'travelplanner_framework_compare' / RUN_TAG
PACK_ROOT = COMPARE_ROOT / 'scientific_pack'
NOTEBOOK_ENV_JSON = PACK_ROOT / 'environment_summary.json'
MAIN_TABLE_MD = PACK_ROOT / 'paper_table_main.md'
MAIN_TABLE_CSV = PACK_ROOT / 'paper_table_main.csv'
SECONDARY_CSV = PACK_ROOT / 'paper_table_secondary.csv'
PAIRWISE_MD = PACK_ROOT / 'pairwise_final_pass_stats.md'
PAIRWISE_JSON = PACK_ROOT / 'pairwise_final_pass_stats.json'
PARETO_CSV = PACK_ROOT / 'pareto_summary.csv'
REPRO_MD = PACK_ROOT / 'reproducibility_report.md'
THREATS_MD = PACK_ROOT / 'threats_to_validity.md'
DSR_MD = PACK_ROOT / 'dsr_episode1_summary.md'

COMPARE_ROOT.mkdir(parents=True, exist_ok=True)
PACK_ROOT.mkdir(parents=True, exist_ok=True)
{
    'run_tag': RUN_TAG,
    'study_root': str(COMPARE_ROOT),
    'pack_root': str(PACK_ROOT),
    'model_name': MODEL_NAME,
    'split': SPLIT,
    'arms': STUDY_ARMS,
    'seeds': SEEDS,
}


{'run_tag': '20260409_233919',
 'study_root': '/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_framework_compare/20260409_233919',
 'pack_root': '/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_framework_compare/20260409_233919/scientific_pack',
 'model_name': 'qwen/qwen3.5-9b',
 'split': 'validation',
 'arms': 'solo_direct,solo_cot,solo_self_refine,planner_executor,langgraph_supervisor,stigmergiagentic',
 'seeds': '42,43,44'}

In [3]:
environment_payload = {
    'run_tag': RUN_TAG,
    'study_root': str(COMPARE_ROOT),
    'repo_root': str(REPO_ROOT),
    'git_sha': git_sha(),
    'utc_started_at': datetime.now(timezone.utc).isoformat(),
    'docker_inputs': {
        'Dockerfile': sha256_file(REPO_ROOT / 'Dockerfile'),
        'docker-compose.yml': sha256_file(REPO_ROOT / 'docker-compose.yml'),
        'requirements.txt': sha256_file(REPO_ROOT / 'requirements.txt'),
    },
    'controlled_dimensions': {
        'provider': 'openrouter',
        'model': MODEL_NAME,
        'base_url': OPENROUTER_BASE_URL,
        'split': SPLIT,
        'temperature': 0.0,
        'request_timeout_seconds': 120,
        'retry_attempts': 2,
        'max_response_tokens': 512,
        'reasoning': {'effort': 'none', 'exclude': True},
    },
    'arms': [
        {'id': 'solo_direct', 'label': 'Direct Solo', 'budget_usd': SOLO_DIRECT_BUDGET},
        {'id': 'solo_cot', 'label': 'CoT Solo', 'budget_usd': SOLO_COT_BUDGET},
        {'id': 'solo_self_refine', 'label': 'Self-Refine Solo', 'budget_usd': SOLO_SELF_REFINE_BUDGET},
        {'id': 'planner_executor', 'label': 'Central Planner-Executor', 'budget_usd': PLANNER_EXECUTOR_BUDGET},
        {'id': 'langgraph_supervisor', 'label': 'Central Graph Supervisor', 'budget_usd': LANGGRAPH_BUDGET},
        {'id': 'stigmergiagentic', 'label': 'StigmergiAgentic', 'budget_usd': OUR_BUDGET},
    ],
    'seeds': [int(item) for item in SEEDS.split(',') if item.strip()],
}
write_json(NOTEBOOK_ENV_JSON, environment_payload)
display(pd.DataFrame(environment_payload['arms']))
environment_payload


$ git rev-parse HEAD
999a29999571d3e9de4e5cfca71bd84011cdd004


,id,label,budget_usd
0,solo_direct,Direct Solo,20.0
1,solo_cot,CoT Solo,20.0
2,solo_self_refine,Self-Refine Solo,20.0
3,planner_executor,Central Planner-Executor,20.0
4,langgraph_supervisor,Central Graph Supervisor,20.0
5,stigmergiagentic,StigmergiAgentic,20.0


{'run_tag': '20260409_233919',
 'study_root': '/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_framework_compare/20260409_233919',
 'repo_root': '/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic',
 'git_sha': '999a29999571d3e9de4e5cfca71bd84011cdd004',
 'utc_started_at': '2026-04-09T23:39:19.161087+00:00',
 'docker_inputs': {'Dockerfile': 'c547da6f6a788cc8d9032600c305a0061613b4853e110bafd02dd3705be8f62f',
  'docker-compose.yml': '22f2b917d2e113010806fd58509e04e63fe000ad80def7ca64269259a40eecd9',
  'requirements.txt': 'ff7f61db8fed4378e9eaebaacc9fa054c1ed2115786f8f356f4fb79507e34163'},
 'controlled_dimensions': {'provider': 'openrouter',
  'model': 'qwen/qwen3.5-9b',
  'base_url': 'https://openrouter.ai/api/v1',
  'split': 'validation',
  'temperature': 0.0,
  'request_timeout_seconds': 120,
  'retry_attempts': 2,
  'max_response_tokens': 512,
  'reasoning': {'effort': 'none', 'exclude': True}},
 'arms': [{'id': 'solo_direct', 'label': 'Direct Solo', 'budge

## Dataset Sanity Checks


In [4]:
ensure_travelplanner_smoke_image(
    log_path=COMPARE_ROOT / 'docker_build.log',
    force_build=DOCKER_FORCE_BUILD,
    skip_build=DOCKER_SKIP_BUILD,
)

run_docker_python(
    ['scripts/setup_travelplanner.py'],
    log_path=COMPARE_ROOT / 'setup_data.log',
)

count_proc = run_docker_python(
    ['-c', "from datasets import load_dataset; ds = load_dataset('osunlp/TravelPlanner', 'validation'); print(len(ds['validation']))"],
    log_path=COMPARE_ROOT / 'dataset_count.log',
)
validation_count = int(count_proc.stdout.strip().splitlines()[-1])
assert validation_count == 180, validation_count
print('validation_count=', validation_count)


$ docker compose build --progress=plain travelplanner-smoke
--progress is a global compose flag, better use `docker compose --progress xx build ...
 Image stigmergiagentic-travelplanner-smoke Building 
#1 [internal] load local bake definitions
#1 reading from stdin 626B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.47kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 0.0s

#4 [internal] load .dockerignore
#4 transferring context: 506B done
#4 DONE 0.0s

#5 [builder 1/5] FROM docker.io/library/python:3.11-slim@sha256:c24e9effa2821a6885165d930d939fec2af0dcf819276138f11dd45e200bd032
#5 resolve docker.io/library/python:3.11-slim@sha256:c24e9effa2821a6885165d930d939fec2af0dcf819276138f11dd45e200bd032 done
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 16.67MB 3.0s done
#6 DONE 3.2s

#7 [builder 3/5] WORKDIR /build
#7 CACHED

#8 [builder 4/5] COPY requirements.txt .
#

## Study Matrix


In [5]:
matrix_rows = []
for arm in [item.strip() for item in STUDY_ARMS.split(',') if item.strip()]:
    for seed in [int(item) for item in SEEDS.split(',') if item.strip()]:
        matrix_rows.append(
            {
                'arm': arm,
                'seed': seed,
                'preflight_queries': PREFLIGHT_COUNT,
                'pilot_queries': PILOT_COUNT,
                'full_queries': FULL_COUNT,
            }
        )
pd.DataFrame(matrix_rows)


,arm,seed,preflight_queries,pilot_queries,full_queries
0,solo_direct,42,3,20,180
1,solo_direct,43,3,20,180
2,solo_direct,44,3,20,180
3,solo_cot,42,3,20,180
4,solo_cot,43,3,20,180
5,solo_cot,44,3,20,180
6,solo_self_refine,42,3,20,180
7,solo_self_refine,43,3,20,180
8,solo_self_refine,44,3,20,180
9,planner_executor,42,3,20,180


## Preflight Gate


In [6]:
if RUN_PREFLIGHT:
    run_docker_python(
        [
            'scripts/run_travelplanner_scientific_study.py',
            '--study-root', f'/app/{COMPARE_ROOT.relative_to(REPO_ROOT).as_posix()}',
            '--provider', 'openrouter',
            '--model', MODEL_NAME,
            '--base-url', OPENROUTER_BASE_URL,
            '--split', SPLIT,
            '--arms', STUDY_ARMS,
            '--seeds', SEEDS,
            '--stage', 'preflight',
            '--preflight-count', str(PREFLIGHT_COUNT),
            '--pilot-count', str(PILOT_COUNT),
            '--full-count', str(FULL_COUNT),
            '--solo-direct-budget-usd', str(SOLO_DIRECT_BUDGET),
            '--solo-cot-budget-usd', str(SOLO_COT_BUDGET),
            '--solo-self-refine-budget-usd', str(SOLO_SELF_REFINE_BUDGET),
            '--planner-executor-budget-usd', str(PLANNER_EXECUTOR_BUDGET),
            '--langgraph-budget-usd', str(LANGGRAPH_BUDGET),
            '--stigmergiagentic-budget-usd', str(OUR_BUDGET),
            '--max-validation-retries', str(LANGGRAPH_MAX_VALIDATION_RETRIES),
            '--our-max-ticks', str(OUR_MAX_TICKS),
            '--our-agents', str(OUR_AGENTS),
        ],
        log_path=COMPARE_ROOT / 'preflight.log',
    )

registry_df = pd.read_csv(PACK_ROOT / 'run_registry.csv')
registry_df[registry_df['stage'] == 'preflight']


$ docker compose run --rm -e PYTHONUNBUFFERED=1 travelplanner-smoke python scripts/run_travelplanner_scientific_study.py --study-root /app/output/travelplanner_framework_compare/20260409_233919 --provider openrouter --model qwen/qwen3.5-9b --base-url https://openrouter.ai/api/v1 --split validation --arms solo_direct,solo_cot,solo_self_refine,planner_executor,langgraph_supervisor,stigmergiagentic --seeds 42,43,44 --stage preflight --preflight-count 3 --pilot-count 20 --full-count 180 --solo-direct-budget-usd 20.0 --solo-cot-budget-usd 20.0 --solo-self-refine-budget-usd 20.0 --planner-executor-budget-usd 20.0 --langgraph-budget-usd 20.0 --stigmergiagentic-budget-usd 20.0 --max-validation-retries 2 --our-max-ticks 30 --our-agents 3
 Container stigmergiagentic-travelplanner-smoke-run-f8d1dadf8a50 Creating 
 Container stigmergiagentic-travelplanner-smoke-run-f8d1dadf8a50 Created 
[preflight] arm=solo_direct seed=42 queries=3
[preflight] arm=solo_cot seed=42 queries=3
[preflight] arm=solo_se

,stage,arm,arm_label,seed,status,failure_kind,failure_message,queries_requested,query_json_count,started_at_utc,ended_at_utc,runtime_wall_seconds,out_dir,config_path,runs_json,official_eval_json,benchmark_summary_json,log_path
0,preflight,solo_direct,Direct Solo,42,success,NaN,NaN,3,3,2026-04-09T23:42:31.383436+00:00,2026-04-09T23:44:03.966519+00:00,92.5823,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
1,preflight,solo_cot,CoT Solo,42,success,NaN,NaN,3,3,2026-04-09T23:44:03.968047+00:00,2026-04-09T23:45:31.545038+00:00,87.5754,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
2,preflight,solo_self_refine,Self-Refine Solo,42,success,NaN,NaN,3,3,2026-04-09T23:45:31.547115+00:00,2026-04-09T23:47:38.288237+00:00,126.7408,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
3,preflight,planner_executor,Central Planner-Executor,42,success,NaN,NaN,3,3,2026-04-09T23:47:38.290082+00:00,2026-04-09T23:49:11.235179+00:00,92.9437,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
4,preflight,langgraph_supervisor,Central Graph Supervisor,42,success,NaN,NaN,3,3,2026-04-09T23:49:11.239169+00:00,2026-04-09T23:52:08.293443+00:00,177.0539,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
5,preflight,stigmergiagentic,StigmergiAgentic,42,success,NaN,NaN,3,3,2026-04-09T23:52:08.295418+00:00,2026-04-09T23:55:33.182150+00:00,204.8849,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...


## Pilot Gate


In [7]:
if RUN_PILOT:
    run_docker_python(
        [
            'scripts/run_travelplanner_scientific_study.py',
            '--study-root', f'/app/{COMPARE_ROOT.relative_to(REPO_ROOT).as_posix()}',
            '--provider', 'openrouter',
            '--model', MODEL_NAME,
            '--base-url', OPENROUTER_BASE_URL,
            '--split', SPLIT,
            '--arms', STUDY_ARMS,
            '--seeds', SEEDS,
            '--stage', 'pilot',
            '--preflight-count', str(PREFLIGHT_COUNT),
            '--pilot-count', str(PILOT_COUNT),
            '--full-count', str(FULL_COUNT),
            '--solo-direct-budget-usd', str(SOLO_DIRECT_BUDGET),
            '--solo-cot-budget-usd', str(SOLO_COT_BUDGET),
            '--solo-self-refine-budget-usd', str(SOLO_SELF_REFINE_BUDGET),
            '--planner-executor-budget-usd', str(PLANNER_EXECUTOR_BUDGET),
            '--langgraph-budget-usd', str(LANGGRAPH_BUDGET),
            '--stigmergiagentic-budget-usd', str(OUR_BUDGET),
            '--max-validation-retries', str(LANGGRAPH_MAX_VALIDATION_RETRIES),
            '--our-max-ticks', str(OUR_MAX_TICKS),
            '--our-agents', str(OUR_AGENTS),
        ],
        log_path=COMPARE_ROOT / 'pilot.log',
    )

registry_df = pd.read_csv(PACK_ROOT / 'run_registry.csv')
registry_df[registry_df['stage'] == 'pilot']


$ docker compose run --rm -e PYTHONUNBUFFERED=1 travelplanner-smoke python scripts/run_travelplanner_scientific_study.py --study-root /app/output/travelplanner_framework_compare/20260409_233919 --provider openrouter --model qwen/qwen3.5-9b --base-url https://openrouter.ai/api/v1 --split validation --arms solo_direct,solo_cot,solo_self_refine,planner_executor,langgraph_supervisor,stigmergiagentic --seeds 42,43,44 --stage pilot --preflight-count 3 --pilot-count 20 --full-count 180 --solo-direct-budget-usd 20.0 --solo-cot-budget-usd 20.0 --solo-self-refine-budget-usd 20.0 --planner-executor-budget-usd 20.0 --langgraph-budget-usd 20.0 --stigmergiagentic-budget-usd 20.0 --max-validation-retries 2 --our-max-ticks 30 --our-agents 3
 Container stigmergiagentic-travelplanner-smoke-run-6f1207ad29c0 Creating 
 Container stigmergiagentic-travelplanner-smoke-run-6f1207ad29c0 Created 
[pilot] arm=solo_direct seed=42 queries=20
[pilot] arm=solo_cot seed=42 queries=20
[pilot] arm=solo_self_refine seed

,stage,arm,arm_label,seed,status,failure_kind,failure_message,queries_requested,query_json_count,started_at_utc,ended_at_utc,runtime_wall_seconds,out_dir,config_path,runs_json,official_eval_json,benchmark_summary_json,log_path
6,pilot,solo_direct,Direct Solo,42,success,NaN,NaN,20,20,2026-04-09T23:55:34.150005+00:00,2026-04-10T00:03:12.091965+00:00,457.9445,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
7,pilot,solo_cot,CoT Solo,42,success,NaN,NaN,20,20,2026-04-10T00:03:12.093479+00:00,2026-04-10T00:11:58.109693+00:00,526.0139,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
8,pilot,solo_self_refine,Self-Refine Solo,42,partial_success,framework_failure,output = runner.run_query(\n ^...,20,7,2026-04-10T00:11:58.111382+00:00,2026-04-10T00:17:21.173984+00:00,323.0614,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
9,pilot,planner_executor,Central Planner-Executor,42,success,NaN,NaN,20,20,2026-04-10T00:17:21.178359+00:00,2026-04-10T00:26:44.877017+00:00,563.6964,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
10,pilot,langgraph_supervisor,Central Graph Supervisor,42,success,NaN,NaN,20,20,2026-04-10T00:26:44.883498+00:00,2026-04-10T00:48:57.540490+00:00,1332.6501,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
11,pilot,stigmergiagentic,StigmergiAgentic,42,success,NaN,NaN,20,20,2026-04-10T00:48:57.542336+00:00,2026-04-10T01:10:15.740691+00:00,1278.2355,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...


## Full Benchmark


In [8]:
if RUN_FULL:
    run_docker_python(
        [
            'scripts/run_travelplanner_scientific_study.py',
            '--study-root', f'/app/{COMPARE_ROOT.relative_to(REPO_ROOT).as_posix()}',
            '--provider', 'openrouter',
            '--model', MODEL_NAME,
            '--base-url', OPENROUTER_BASE_URL,
            '--split', SPLIT,
            '--arms', STUDY_ARMS,
            '--seeds', SEEDS,
            '--stage', 'full',
            '--preflight-count', str(PREFLIGHT_COUNT),
            '--pilot-count', str(PILOT_COUNT),
            '--full-count', str(FULL_COUNT),
            '--solo-direct-budget-usd', str(SOLO_DIRECT_BUDGET),
            '--solo-cot-budget-usd', str(SOLO_COT_BUDGET),
            '--solo-self-refine-budget-usd', str(SOLO_SELF_REFINE_BUDGET),
            '--planner-executor-budget-usd', str(PLANNER_EXECUTOR_BUDGET),
            '--langgraph-budget-usd', str(LANGGRAPH_BUDGET),
            '--stigmergiagentic-budget-usd', str(OUR_BUDGET),
            '--max-validation-retries', str(LANGGRAPH_MAX_VALIDATION_RETRIES),
            '--our-max-ticks', str(OUR_MAX_TICKS),
            '--our-agents', str(OUR_AGENTS),
        ],
        log_path=COMPARE_ROOT / 'full.log',
    )

registry_df = pd.read_csv(PACK_ROOT / 'run_registry.csv')
registry_df[registry_df['stage'] == 'full']


$ docker compose run --rm -e PYTHONUNBUFFERED=1 travelplanner-smoke python scripts/run_travelplanner_scientific_study.py --study-root /app/output/travelplanner_framework_compare/20260409_233919 --provider openrouter --model qwen/qwen3.5-9b --base-url https://openrouter.ai/api/v1 --split validation --arms solo_direct,solo_cot,solo_self_refine,planner_executor,langgraph_supervisor,stigmergiagentic --seeds 42,43,44 --stage full --preflight-count 3 --pilot-count 20 --full-count 180 --solo-direct-budget-usd 20.0 --solo-cot-budget-usd 20.0 --solo-self-refine-budget-usd 20.0 --planner-executor-budget-usd 20.0 --langgraph-budget-usd 20.0 --stigmergiagentic-budget-usd 20.0 --max-validation-retries 2 --our-max-ticks 30 --our-agents 3
 Container stigmergiagentic-travelplanner-smoke-run-5a59669818f9 Creating 
 Container stigmergiagentic-travelplanner-smoke-run-5a59669818f9 Created 
[full] arm=solo_direct seed=42 queries=180
[full] arm=solo_direct seed=43 queries=180
[full] arm=solo_direct seed=44 

,stage,arm,arm_label,seed,status,failure_kind,failure_message,queries_requested,query_json_count,started_at_utc,ended_at_utc,runtime_wall_seconds,out_dir,config_path,runs_json,official_eval_json,benchmark_summary_json,log_path
12,full,solo_direct,Direct Solo,42,success,NaN,NaN,180,180,2026-04-10T01:10:16.578108+00:00,2026-04-10T02:20:30.403422+00:00,4213.7911,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
13,full,solo_direct,Direct Solo,43,success,NaN,NaN,180,180,2026-04-10T02:20:30.405470+00:00,2026-04-10T03:27:08.736679+00:00,3998.3112,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
14,full,solo_direct,Direct Solo,44,success,NaN,NaN,180,180,2026-04-10T03:27:08.739194+00:00,2026-04-10T04:33:45.928183+00:00,3997.1554,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
15,full,solo_cot,CoT Solo,42,success,NaN,NaN,180,180,2026-04-10T04:33:45.929830+00:00,2026-04-10T05:39:40.636753+00:00,3954.6397,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
16,full,solo_cot,CoT Solo,43,success,NaN,NaN,180,180,2026-04-10T05:39:40.638410+00:00,2026-04-10T06:47:16.529458+00:00,4055.8704,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
17,full,solo_cot,CoT Solo,44,success,NaN,NaN,180,180,2026-04-10T06:47:16.531736+00:00,2026-04-10T07:52:22.928022+00:00,3906.4327,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
18,full,planner_executor,Central Planner-Executor,42,partial_success,framework_failure,output = runner.run_query(\n ^...,180,111,2026-04-10T07:52:22.932141+00:00,2026-04-10T08:41:30.350859+00:00,2947.3949,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
19,full,planner_executor,Central Planner-Executor,43,partial_success,framework_failure,output = runner.run_query(\n ^...,180,111,2026-04-10T08:41:30.367004+00:00,2026-04-10T09:33:57.419449+00:00,3146.9651,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...,/app/output/travelplanner_framework_compare/20...
20,full,planner_executor,Central Planner-Executor,44,partial_success,framework_failure,output = runner.run_query(\n ^...,180,111,2026-04-10T09:33:57.425881+00:00,2026-04-1

## Official Scores


In [9]:
if BUILD_PACK:
    run_docker_python(
        [
            'scripts/build_travelplanner_scientific_pack.py',
            '--study-root', f'/app/{COMPARE_ROOT.relative_to(REPO_ROOT).as_posix()}',
            '--canonical-seed', '42',
        ],
        log_path=COMPARE_ROOT / 'build_scientific_pack.log',
    )

for path in [
    MAIN_TABLE_MD,
    MAIN_TABLE_CSV,
    SECONDARY_CSV,
    PAIRWISE_MD,
    PAIRWISE_JSON,
    PARETO_CSV,
    REPRO_MD,
    THREATS_MD,
    DSR_MD,
]:
    require_path(path)


$ docker compose run --rm -e PYTHONUNBUFFERED=1 travelplanner-smoke python scripts/build_travelplanner_scientific_pack.py --study-root /app/output/travelplanner_framework_compare/20260409_233919 --canonical-seed 42
 Container stigmergiagentic-travelplanner-smoke-run-bf725957d611 Creating 
 Container stigmergiagentic-travelplanner-smoke-run-bf725957d611 Created 
{
  "main_table_md": "/app/output/travelplanner_framework_compare/20260409_233919/scientific_pack/paper_table_main.md",
  "main_table_csv": "/app/output/travelplanner_framework_compare/20260409_233919/scientific_pack/paper_table_main.csv",
  "secondary_csv": "/app/output/travelplanner_framework_compare/20260409_233919/scientific_pack/paper_table_secondary.csv",
  "pairwise_json": "/app/output/travelplanner_framework_compare/20260409_233919/scientific_pack/pairwise_final_pass_stats.json",
  "pairwise_md": "/app/output/travelplanner_framework_compare/20260409_233919/scientific_pack/pairwise_final_pass_stats.md",
  "pareto_csv": "/

In [10]:
render_markdown_file(MAIN_TABLE_MD)
pd.read_csv(MAIN_TABLE_CSV)


# Paper Table — Main Results

## Valid Arms

| Philosophy | Valid Runs | Delivery | Commonsense Micro | Commonsense Macro | Hard Constraint Micro | Hard Constraint Macro | Final Pass |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Direct Solo | 3/3 | 58.0 ± 0.6 | 45.9 ± 0.6 | 15.0 ± 0.6 | 19.8 ± 0.7 | 12.2 ± 0.6 | 4.1 ± 0.6 |
| CoT Solo | 3/3 | 50.9 ± 0.6 | 42.1 ± 0.3 | 17.6 ± 0.8 | 21.4 ± 0.6 | 11.9 ± 0.6 | 5.7 ± 0.3 |
| StigmergiAgentic | 3/3 | 55.4 ± 0.3 | 43.4 ± 0.3 | 15.2 ± 1.2 | 22.8 ± 0.5 | 14.1 ± 0.6 | 8.5 ± 0.6 |

## Invalid or Failed Arms

| Philosophy | Status | Successful Full Runs | Failure Note |
| --- | --- | --- | --- |
| Self-Refine Solo | failed | 0/3 |  |
| Central Planner-Executor | failed | 0/3 | output = runner.run_query(              ^^^^^^^^^^^^^^^^^   File "/app/adapters/travelplanner/scientific_baselines.py",  |
| Central Graph Supervisor | partial_success | 2/3 | response = self.llm_client.call(                ^^^^^^^^^^^^^^^^^^^^^   File "/app/llm/client.py", line 280, in call     |


,arm,arm_label,status,valid_runs,expected_runs,failure_note,delivery_rate_mean,delivery_rate_sd,commonsense_micro_mean,commonsense_micro_sd,...,tokens_total_mean,tokens_total_sd,cost_total_usd_mean,cost_total_usd_sd,runtime_wall_seconds_mean,runtime_wall_seconds_sd,avg_runtime_per_query_seconds_mean,avg_runtime_per_query_seconds_sd,avg_coordination_overhead_mean,avg_coordination_overhead_sd
0,solo_direct,Direct Solo,valid,3,3,NaN,0.579630,0.006415,0.458796,0.005824,...,4.020847e+05,79.607370,0.032008,0.000755,4069.6470,124.733327,22.514333,0.693749,1.00,0.000000
1,solo_cot,CoT Solo,valid,3,3,NaN,0.509259,0.006415,0.421296,0.003282,...,3.859283e+05,1644.965147,0.029455,0.000167,3972.2144,76.269357,21.972633,0.424008,1.00,0.000000
2,solo_self_refine,Self-Refine Solo,failed,0,3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,planner_executor,Central Planner-Executor,failed,0,3,output = runner.run_query(\n ^^^^^...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,langgraph_supervisor,Central Graph Supervisor,partial_success,2,3,response = self.llm_client.call(\n ...,0.580556,0.011785,0.408333,0.002946,...,2.841506e+06,7337.647068,0.245271,0.012349,14321.7832,1152.114817,79.470950,6.395852,24.71,0.028284
5,stigmergiagentic,StigmergiAgentic,valid,3,3,NaN,0.553704,0.003208,0.434259,0.002807,...,1.635611e+06,4131.071169,0.137402,0.006867,12917.5340,545.359493,71.666200,3.030722,15.84,0.580431


## Paired Statistical Analysis


In [11]:
render_markdown_file(PAIRWISE_MD)
load_json(PAIRWISE_JSON)


# Pairwise Final-Pass Statistics

| Left | Right | Wins/Losses/Ties | Delta Final Pass | McNemar p | CI Low | CI High |
| --- | --- | --- | --- | --- | --- | --- |
| StigmergiAgentic | Direct Solo | 9/3/168 | 3.3 | 0.1460 | -0.6 | 7.2 |
| StigmergiAgentic | CoT Solo | 6/2/172 | 2.2 | 0.2891 | -0.6 | 5.6 |
| StigmergiAgentic | solo_self_refine | NA | NA | NA | NA | NA |
| StigmergiAgentic | planner_executor | NA | NA | NA | NA | NA |
| StigmergiAgentic | Central Graph Supervisor | 13/3/164 | 5.6 | 0.0213 | 1.7 | 10.0 |


{'rows': [{'left': 'StigmergiAgentic',
   'right': 'Direct Solo',
   'available': True,
   'paired_queries': 180,
   'wins': 9,
   'losses': 3,
   'ties': 168,
   'left_rate': 0.07777777777777778,
   'right_rate': 0.044444444444444446,
   'delta_final_pass_rate': 0.03333333333333333,
   'mcnemar_exact_p': 0.14599609375,
   'bootstrap_ci_low': -0.005555555555555556,
   'bootstrap_ci_high': 0.07222222222222222},
  {'left': 'StigmergiAgentic',
   'right': 'CoT Solo',
   'available': True,
   'paired_queries': 180,
   'wins': 6,
   'losses': 2,
   'ties': 172,
   'left_rate': 0.07777777777777778,
   'right_rate': 0.05555555555555555,
   'delta_final_pass_rate': 0.022222222222222227,
   'mcnemar_exact_p': 0.2890625,
   'bootstrap_ci_low': -0.005555555555555556,
   'bootstrap_ci_high': 0.05555555555555555},
  {'left': 'StigmergiAgentic',
   'right': 'solo_self_refine',
   'available': False,
   'reason': 'missing_successful_canonical_run'},
  {'left': 'StigmergiAgentic',
   'right': 'planner

## Operational Analysis


In [12]:
pd.read_csv(PARETO_CSV)


,arm,arm_label,status,final_pass_rate_mean,cost_total_usd_mean,runtime_wall_seconds_mean,avg_coordination_overhead_mean
0,solo_direct,Direct Solo,valid,0.040741,0.032008,4069.6470,1.00
1,solo_cot,CoT Solo,valid,0.057407,0.029455,3972.2144,1.00
2,solo_self_refine,Self-Refine Solo,failed,NaN,NaN,NaN,NaN
3,planner_executor,Central Planner-Executor,failed,NaN,NaN,NaN,NaN
4,langgraph_supervisor,Central Graph Supervisor,partial_success,0.022222,0.245271,14321.7832,24.71
5,stigmergiagentic,StigmergiAgentic,valid,0.085185,0.137402,12917.5340,15.84


## Reproducibility and Failures


In [13]:
render_markdown_file(REPRO_MD)


# Reproducibility Report

- Study root: `/app/output/travelplanner_framework_compare/20260409_233919`
- Canonical seed for paired analysis: `42`
- Valid full arms: `3/6`

## Status Counts

- `full::partial_success`: 4
- `full::success`: 11
- `pilot::partial_success`: 1
- `pilot::success`: 5
- `preflight::success`: 6

## Failure Notes

- `pilot` / `Self-Refine Solo` / seed `42`: partial_success (framework_failure)
- `full` / `Central Planner-Executor` / seed `42`: partial_success (framework_failure)
- `full` / `Central Planner-Executor` / seed `43`: partial_success (framework_failure)
- `full` / `Central Planner-Executor` / seed `44`: partial_success (framework_failure)
- `full` / `Central Graph Supervisor` / seed `43`: partial_success (infra_failure)


## DSR / FEDS Pack


In [14]:
render_markdown_file(DSR_MD)
render_markdown_file(THREATS_MD)


# DSR Episode 1 Summary

This scientific pack operationalizes OC3 and FEDS Episode 1 as a controlled same-backbone benchmark across organization philosophies on TravelPlanner.
- Valid arms in the principal table: `3/6`
- Replications per valid arm targeted: `3`
- Primary criterion: `Final Pass Rate`
- Best valid arm by mean final pass: `StigmergiAgentic` (8.5 ± 0.6)

Interpretation should remain cautious: the study compares reproducible organization philosophies under a controlled TravelPlanner protocol, not universal framework superiority across all domains.


# Threats to Validity

- Internal validity: prompt engineering and baseline-specific decomposition choices can influence results, even under controlled backbone/provider settings.
- Construct validity: TravelPlanner primarily measures constrained itinerary planning, not the full space of software-engineering coordination tasks.
- External validity: results on TravelPlanner do not automatically transfer to code migration or general enterprise work.
- Conclusion validity: the study uses a controlled protocol with three replications, which improves robustness but does not constitute an industrial field trial.
- Provider validity: OpenRouter and upstream serving variability can affect latency, malformed outputs, and run reproducibility.
